# Preprocessing Pipeline for Apartment Rent Data

## 1. Project Overview and Data Loading

This project analyzes apartment rental data to explore how pricing varies with location, property characteristics, and amenities. The notebook focuses on data cleaning, preprocessing, sampling methods, feature engineering, and exploratory analysis to prepare the dataset for future modeling or deeper investigation.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%config InlineBackend.figure_formats = ['svg']

The dataset is loaded and inspected to understand its structure, size, and initial content before any cleaning or transformation.

In [ ]:
data = pd.read_csv('apartment_rent.csv')
data.head()

In [ ]:
data.shape
print(f"Dataset shape: {data.shape}")

## 2. Data Cleaning and Preparation

Duplicate records are checked and removed to prevent repeated observations from affecting the analysis.

In [ ]:
# Check for duplicate records
print('Number of duplicates:', data.duplicated().sum())

In [ ]:
# Remove duplicate records
data = data.drop_duplicates() 
print('Number of duplicates after removal:', data.duplicated().sum())

The "title" column is removed because it is not required for the analysis and does not contribute meaningful numerical or categorical information.

In [ ]:
data = data.drop(columns = ['title'])
data.head()

Missing values are identified and handled using a combination of targeted imputation and row removal for critical fields.

In [ ]:
missing_values = data.isnull()

# Print the number of NaNs in each column
print("Number of NaN values in each column:")
print(missing_values.sum())

In [ ]:
data['amenities'] = data['amenities'].fillna('none advertised')
data['pets_allowed'] = data['pets_allowed'].fillna('No')

In [ ]:
data['bathrooms'] = data['bathrooms'].fillna(data['bathrooms'].mode()[0])
data['bedrooms'] = data['bedrooms'].fillna(data['bedrooms'].mode()[0])

In [ ]:
data = data.dropna(subset = ['cityname', 'latitude', 'price'])

The "time" column is converted to a datetime format so that temporal features can be extracted for analysis.

In [ ]:
data['time'] = pd.to_datetime(data['time'], unit='s')
data.head()

Year, month, and day-of-week features are extracted from the timestamp to support time-based exploration. The original timestamp column is then removed to keep the dataset compact.

In [ ]:
# extract the Year out of the date
data['year'] = data['time'].dt.year

# extract the month out of the date
data['month'] = data['time'].dt.month

# extract the day out of the date
data['day_of_week'] = data['time'].dt.day_name()

# Drop the 'time' column
data = data.drop(columns = ['time'])

data.head()

Price outliers are examined using a boxplot and then handled with the IQR method to reduce the influence of extreme values.

In [ ]:
sns.boxplot(x = data['price'], color = 'salmon')

plt.title('Price Boxplot')
plt.xlabel('Price')
plt.show()

In [ ]:
# Delete outliers of "price" column using IQR
Q1 = data['price'].quantile(0.25)
Q3 = data['price'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

data = data[(data['price'] >= lower_bound) & (data['price'] <= upper_bound)]

plt.figure(figsize=(10, 5))
sns.boxplot(x=data['price'], color='green')
plt.title('Price Boxplot after noise removal')
plt.xlabel('Price')
plt.show()

Removing extreme price values helps the analysis focus on the central distribution of rents.

In [ ]:
data.reset_index(drop=True, inplace=True)
data.info()

## 3. Sampling Methods

Three sampling methods are applied to the cleaned dataset: simple random sampling, systematic sampling, and stratified sampling. Their outputs are compared to evaluate how well each method preserves the city distribution.

In [ ]:
# 1. Simple Random Sampling
random_sample = data.sample(n = int(len(data) * 0.1), random_state = 42)

# 2. Systematic Sampling
systematic_sample = data.iloc[::10]

# 3. Stratified Sampling by city
stratified_samples = []

for city, group in data.groupby('cityname'):
    n_samples = max(1, int(len(group) * 0.1))
    sample = group.sample(n = n_samples, random_state = 42)
    stratified_samples.append(sample)

stratified_sample = pd.concat(stratified_samples, axis = 0)

print(f"Stratified sample size: {len(stratified_sample)}")
print(f"Number of unique cities in sample: {stratified_sample['cityname'].nunique()}")

# Compare Results
print(f"Original data: {len(data):,} records")
print(f"Random sample: {len(random_sample):,} records ({len(random_sample)/len(data)*100:.1f}%)")
print(f"Systematic sample: {len(systematic_sample):,} records")
print(f"Stratified sample: {len(stratified_sample):,} records")

The city distribution in each sample is compared against the original dataset to measure sampling quality. Average and maximum absolute differences are used as simple evaluation metrics.

In [ ]:
# Compare city proportions for all 3 sampling methods

# Original city proportions
orig_props = data['cityname'].value_counts(normalize=True) * 100

# Proportions for each sample
random_props = random_sample['cityname'].value_counts(normalize=True) * 100
systematic_props = systematic_sample['cityname'].value_counts(normalize=True) * 100
stratified_props = stratified_sample['cityname'].value_counts(normalize=True) * 100

# Create comparison DataFrame
city_comparison = pd.DataFrame({
    'Original_%': orig_props,
    'Random_%': random_props,
    'Systematic_%': systematic_props,
    'Stratified_%': stratified_props
}).fillna(0)

# Calculate differences
city_comparison['Random_Diff'] = city_comparison['Random_%'] - city_comparison['Original_%']
city_comparison['Systematic_Diff'] = city_comparison['Systematic_%'] - city_comparison['Original_%']
city_comparison['Stratified_Diff'] = city_comparison['Stratified_%'] - city_comparison['Original_%']

print("\n" + "="*70)
print("City Proportions Comparison (Top 10 cities)")
print("="*70)
print(city_comparison[['Original_%', 'Random_%', 'Systematic_%', 'Stratified_%']].head(10).round(2))

# Quality metrics for each method
print("\n" + "="*70)
print("Sampling Quality - Average Absolute Difference (%)")
print("="*70)
print(f"Random sampling:      {city_comparison['Random_Diff'].abs().mean():.3f}%")
print(f"Systematic sampling:  {city_comparison['Systematic_Diff'].abs().mean():.3f}%")
print(f"Stratified sampling:  {city_comparison['Stratified_Diff'].abs().mean():.3f}%")

print("\n" + "="*70)
print("Sampling Quality - Maximum Absolute Difference (%)")
print("="*70)
print(f"Random sampling:      {city_comparison['Random_Diff'].abs().max():.3f}%")
print(f"Systematic sampling:  {city_comparison['Systematic_Diff'].abs().max():.3f}%")
print(f"Stratified sampling:  {city_comparison['Stratified_Diff'].abs().max():.3f}%")

# Show which method performs best
best_avg = city_comparison[['Random_Diff', 'Systematic_Diff', 'Stratified_Diff']].abs().mean().idxmin()
best_max = city_comparison[['Random_Diff', 'Systematic_Diff', 'Stratified_Diff']].abs().max().idxmin()
print("\n" + "="*70)
print("Best performer:")
print(f"Lowest average difference: {best_avg.replace('_Diff','')}")
print(f"Lowest maximum difference: {best_max.replace('_Diff','')}")

Sampling methods produce different levels of fidelity to the original city distribution, with stratified sampling generally preserving representation better than the others.

## 4. Exploratory Data Analysis

The correlation matrix is used to examine relationships between numeric variables and identify possible associations worth exploring further.

In [ ]:
# Plot correlation matrix
corr_matrix = data.corr(numeric_only=True)

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix of Numeric Features')
plt.show()

There are relatively strong correlations between: bathrooms and square_feet, bathrooms and bedrooms, bedrooms and square_feet. 

Rental price per square foot is calculated to make location and pricing comparisons more informative.

In [ ]:
data['pps'] = data['price'] / data['square_feet']
data.head()

The ten states with the highest average price per square foot are identified to highlight regional differences in rental pricing.

In [ ]:
state_avg_pps = data.groupby('state')['pps'].mean().reset_index()
top_10_states = state_avg_pps.sort_values(by='pps', ascending=False).head(10)

plt.figure(figsize=(10, 6))
plt.bar(top_10_states['state'], top_10_states['pps'], color='skyblue')
plt.xlabel('State')
plt.ylabel('Average price per square foot')
plt.title('Top 10 States by Average Price per Square Foot')
plt.show()

Dallas is examined in more detail to explore how rental price per square foot varies with location and property size.

In [ ]:
dallas = data[data['cityname'] == 'Dallas'].copy()
dallas.head()

In [ ]:
fig, ax = plt.subplots(figsize = (9, 7))

scatter = ax.scatter(
    dallas['longitude'],
    dallas['latitude'],
    s=10,
    c=dallas['pps'],
    cmap='magma',
    alpha=0.6
)

cbar = fig.colorbar(scatter, ax=ax)
cbar.set_label('pps')

dallas_center_lon = -96.79708
dallas_center_lat = 32.77653

ax.scatter(
    dallas_center_lon,
    dallas_center_lat,
    c='yellow',
    s=100,
    marker='X',
    label='center of Dallas'
)

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Dallas: pps vs Location')
ax.legend()

Properties closer to the city center appear to have higher price-per-square-foot values, suggesting a location effect on rent.

In [ ]:
# Examine the relationship between house size and rental price in Dallas
sns.jointplot(
    data = dallas,
    x = 'square_feet',
    y = 'price', 
    kind = 'scatter', 
    height = 6
)

plt.show()

There is a relatively strong relation between square_feet and price in dallas state. 

The distribution of rental prices is compared between properties with and without elevator access.

In [ ]:
# The impact of an elevator on price
fig, ax = plt.subplots(figsize=(10, 6)) 

data['has_elevator'] = data['amenities'].str.contains('Elevator', case=False, na=False)

data_elev = data[data['has_elevator'] == True]
data_no_elev = data[data['has_elevator'] == False]

sns.kdeplot(data_elev['price'], label='with elevator', ax=ax)
sns.kdeplot(data_no_elev['price'], label='without elevator', ax=ax)

ax.legend()
plt.show()

Properties with elevators tend to show a slightly higher price distribution, although the overlap suggests that elevator access is only one of several factors affecting rent.

Rental price distributions in Washington and Texas are compared using a violin plot to highlight differences in spread and concentration.

In [ ]:
states = ['WA', 'TX']
filtered_df = data[data['state'].isin(states)]

fig, ax = plt.subplots(figsize=(10, 6)) 

sns.violinplot(x='state', y='price', data=filtered_df, ax=ax)

plt.show()

Texas shows a larger concentration of lower-priced homes, while Washington appears to have a higher overall price distribution.

## 5. Conclusion

The analysis shows that rental price is influenced by several factors, including location, square footage, and selected amenities. The preprocessing steps make the dataset more reliable for further analysis, while the exploratory visualizations highlight useful patterns across cities and states. Stratified sampling also appears to preserve the original city distribution more effectively than the other sampling methods in this notebook.

## 6. Future Work

Future work could include building a predictive model for rental price using the cleaned dataset. Additional analysis could also examine more detailed regional effects, test feature importance, and compare model performance across different algorithms.

